# Data Drift Guardian — воспроизводимая демонстрация

Notebook создаёт независимые Reference/Current, проверяет файловую загрузку, вызывает единый `analyze()` и сравнивает контроль, числовой сдвиг, изменение категорий и рост пропусков. Все таблицы ниже возникают из выполненного кода. Дедлайн всего проекта — 27.09.2026.

## 1. Окружение и конфигурация

Запускайте notebook из корня репозитория после `python -m pip install -r requirements.txt`. Код не использует личные абсолютные пути.

In [ ]:
from copy import deepcopy
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import json
import platform
import subprocess
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    candidates = [parent for parent in PROJECT_ROOT.parents if (parent / "pyproject.toml").is_file()]
    if not candidates:
        raise RuntimeError("Не найден корень репозитория с pyproject.toml")
    PROJECT_ROOT = candidates[0]

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_drift_guardian import analyze
from data_drift_guardian.config import load_config
from data_drift_guardian.ingestion import load_table
from data_drift_guardian.reporting import distribution_figure
from scripts.generate_demo_data import generate_demo_data, write_generated_data

commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
versions = {
    "Python": platform.python_version(),
    "NumPy": version("numpy"),
    "pandas": version("pandas"),
    "SciPy": version("scipy"),
    "scikit-learn": version("scikit-learn"),
    "LightGBM": version("lightgbm"),
}
display(pd.DataFrame(versions.items(), columns=["Компонент", "Версия"]))
print("Commit:", commit)

In [ ]:
config = load_config(PROJECT_ROOT / "configs" / "default.yaml")
experiment_config = deepcopy(config)
experiment_config["drift"]["distance_thresholds"].update(
    {"psi": 0.10, "js": 0.05}
)
experiment_config["adversarial"].update(
    {"enabled": True, "roc_auc_threshold": 0.70}
)
print(json.dumps(experiment_config, ensure_ascii=False, indent=2))

Пороги PSI `0.10`, JS `0.05` и ROC-AUC `0.70` используются только как исследовательские правила демонстрации. Порог Wasserstein оставлен `None`, потому что расстояние измеряется в единицах признака и единая граница для возраста и дохода некорректна.

## 2. Генерация и файловая загрузка

Сценарий `combined` вносит известный сдвиг возраста на 8 лет, меняет доли регионов и повышает долю пропусков income с 2% до 8%. Reference и Current всё равно генерируются независимо.

In [ ]:
N_REFERENCE = 2_000
N_CURRENT = 1_200
DEMO_SEED = 42

generated = generate_demo_data(
    seed=DEMO_SEED,
    n_reference=N_REFERENCE,
    n_current=N_CURRENT,
    scenario="combined",
)
temporary_data = TemporaryDirectory(prefix="data-drift-demo-")
paths = write_generated_data(
    generated,
    output_dir=Path(temporary_data.name),
    formats=("csv", "parquet"),
)
reference = load_table(Path(temporary_data.name) / paths["reference_parquet"])
current = load_table(Path(temporary_data.name) / paths["current_parquet"])

print("Reference shape:", reference.shape)
print("Current shape:", current.shape)
print("Independent batches:", generated.metadata["independent_batches"])
display(reference.head())
display(current.head())

## 3. Единый API и JSON-контракт

In [ ]:
started = perf_counter()
combined_result = analyze(reference, current, config=experiment_config)
combined_seconds = perf_counter() - started
json.dumps(combined_result, allow_nan=False)

print("Время анализа, с:", round(combined_seconds, 4))
display(pd.DataFrame([combined_result["summary"]]))
display(pd.DataFrame(combined_result["alerts"]))

## 4. Schema Validation и Data Quality

Рост пропусков измеряется в процентных пунктах: переход от 2% к 8% равен 6 п.п., а не 6%.

In [ ]:
quality_rows = []
for feature, checks in combined_result["quality"]["feature_checks"].items():
    for check in checks:
        quality_rows.append(
            {
                "feature": feature,
                "check": check["name"],
                "value": check["value"],
                "threshold": check["threshold"],
                "status": check["status"],
                "alert": check["alert"],
            }
        )
display(pd.DataFrame(quality_rows))

## 5. Статистические методы и Adversarial Validation

KS и χ² принимают решение по BH-скорректированному p-value. PSI/JS используют исследовательские пороги. Wasserstein вычисляется без общего алерта.

In [ ]:
drift_rows = []
for feature, feature_result in combined_result["drift"]["features"].items():
    for method, check in feature_result["checks"].items():
        drift_rows.append(
            {
                "feature": feature,
                "method": method,
                "value": check["value"],
                "p_value": check["p_value"],
                "adjusted_p_value": check["adjusted_p_value"],
                "threshold": check["threshold"],
                "status": check["status"],
                "alert": check["alert"],
            }
        )
display(pd.DataFrame(drift_rows))
display(pd.DataFrame(
    {
        "fold": range(1, len(combined_result["adversarial"]["fold_auc"]) + 1),
        "roc_auc": combined_result["adversarial"]["fold_auc"],
    }
))
print("OOF ROC-AUC:", combined_result["adversarial"]["roc_auc"])
print("Feature importance:", combined_result["adversarial"]["feature_importance"])

## 6. Сравнение сценариев

Контроль запускается на пяти seed, чтобы первично увидеть ложные тревоги. Это не полноценная калибровка.

In [ ]:
def check(result, feature, method):
    return result["drift"]["features"][feature]["checks"][method]

def quality_value(result, feature, name):
    return next(
        item["value"]
        for item in result["quality"]["feature_checks"][feature]
        if item["name"] == name
    )

def run_scenario(scenario, seed):
    data = generate_demo_data(
        seed=seed,
        n_reference=N_REFERENCE,
        n_current=N_CURRENT,
        scenario=scenario,
    )
    started = perf_counter()
    result = analyze(data.reference, data.current, config=deepcopy(experiment_config))
    elapsed = perf_counter() - started
    return result, {
        "scenario": scenario,
        "seed": seed,
        "n_reference": N_REFERENCE,
        "n_current": N_CURRENT,
        "status": result["summary"]["status"],
        "alerts": result["summary"]["n_alerts"],
        "age_ks_q": check(result, "age", "ks")["adjusted_p_value"],
        "age_wasserstein": check(result, "age", "wasserstein")["value"],
        "age_psi": check(result, "age", "psi")["value"],
        "age_js": check(result, "age", "js")["value"],
        "region_chi2_q": check(result, "region", "chi2")["adjusted_p_value"],
        "region_psi": check(result, "region", "psi")["value"],
        "region_js": check(result, "region", "js")["value"],
        "income_missing_increase_pp": quality_value(result, "income", "missing_increase_pp"),
        "oof_roc_auc": result["adversarial"]["roc_auc"],
        "seconds": elapsed,
        "commit": commit,
    }

scenario_results = {}
records = []
for seed in [7, 42, 2026, 9001, 1309]:
    result, record = run_scenario("none", seed)
    scenario_results[("none", seed)] = result
    records.append(record)
for scenario in ["numeric", "categorical", "missingness", "combined"]:
    result, record = run_scenario(scenario, 42)
    scenario_results[(scenario, 42)] = result
    records.append(record)

experiment_table = pd.DataFrame(records)
display(experiment_table)

In [ ]:
control_table = experiment_table[experiment_table["scenario"] == "none"]
print("Контрольных запусков с алертами:", int((control_table["alerts"] > 0).sum()), "из", len(control_table))
display(control_table[["seed", "alerts", "age_ks_q", "region_chi2_q", "oof_roc_auc"]])

## 7. Те же распределения, которые показывает дашборд

In [ ]:
distribution_figure(reference["age"], current["age"], "numeric").show()
distribution_figure(reference["region"], current["region"], "categorical").show()

## 8. Выводы первой недели

- Контроль сравнивает независимые батчи, а не таблицу с её копией. Поэтому отдельное ложное срабатывание возможно и должно учитываться при калибровке.
- Числовой сценарий должен проявляться прежде всего в `age`; категориальный — в `region`; missingness — в Data Quality для `income`.
- Высокий OOF ROC-AUC означает, что LightGBM различает источники, но не доказывает падение качества основной модели.
- Отсутствие алерта не доказывает идентичность всех распределений: тесты обладают ограниченной мощностью и проверяют конкретные аспекты.
- Data Quality отделяет нарушения контракта данных от валидного изменения распределения.
- Gain-важность показывает вклад в классификацию источника и не является причинной оценкой.
- Пороги PSI, JS и ROC-AUC остаются исследовательскими; порог Wasserstein должен задаваться отдельно с учётом единиц и бизнес-смысла признака.